In [1]:
import os
import pandas as pd
from pathlib import Path
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (Multitox)

This notebook curates the **Multitox** dataset from a CSV source that encodes toxin annotations as a multi-class label. We standardize the input, derive multiple binary datasets (general toxin vs non-toxin, plus subtype-specific sets such as neurotoxic/cytotoxic/hemotoxic), perform duplicate consistency checks for each derived dataset, and export curated outputs and metadata.

- **Toxic effect / endpoint:** hemolytic, neurotoxic, cytotoxic
- **Source:** Multitox
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads the raw Multitox table** (`toxin3052.csv`) which contains:
  - `sequence`
  - a multi-class `label` with the following semantics:
    - `0` → non-toxin
    - `1` → neurotoxin
    - `2` → cytotoxin
    - `3` → hemotoxin
    - `4` → enterotoxin
- **Derives task-specific datasets**:
  - **General toxin (binary)**: collapses all toxin subclasses `{1,2,3,4}` into `label = 1` and keeps `label = 0` for non-toxins.
  - **Subtype views**:
    - `neurotoxic`: selects `label == 1` (positive-only subset).
    - `cytotoxic`: selects `label == 2` and remaps to `label = 1` (positive-only subset).
    - `hemotoxic`: selects `label == 3` and remaps to `label = 1` (positive-only subset).
- **Checks duplicated sequences** independently for:
  - the general toxin binary dataset,
  - each subtype-specific subset.
  Duplicates are collapsed when consistent; conflicting cases are reported as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated outputs**:
  - `processed_neurotoxic_dataset.csv` (positive-only),
  - `processed_cytotoxic_dataset.csv` (positive-only),
  - `processed_hemolytic_dataset.csv` (hemotoxic, positive-only),
  - `detected_error_sequences.csv`,
  - `metadata.json`.

In [2]:
name_source = "Multitox"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df = pd.read_csv(f"{PATH_INPUT}/{name_source}/toxin3052.csv", skiprows=1, names=["sequence", "label"])  # 0 -> no toxin; 1 -> neurotoxin; 2 -> cytotoxin; 3 -> hemotoxin; 4-> enterotoxins

- Split dataset by activity

In [4]:
df_neurotoxic = df[df['label'] == 1]

In [5]:
df_cytotoxic = (df[df['label'] == 2]
                .replace(2, 1))

In [6]:
df_hemotoxic = (df[df['label'] == 3]
                .replace(3, 1))

In [7]:
# Asignar valor 1 para las toxinas y 0 para los no tóxicos
df['label'] = df['label'].apply(lambda x: 1 if x in [1, 2, 3, 4] else 0)

- Checking duplicates

In [8]:
df_remove_duplicated_cytotoxic, df_errors_cytotoxic, df_unique_cytotoxic = processing_duplicated(df_cytotoxic, group_seq="sequence", sort_key="label")

In [9]:
df_remove_duplicated_hemotoxic, df_errors_hemotoxic, df_unique_hemotoxic = processing_duplicated(df_hemotoxic, group_seq="sequence", sort_key="label")

In [10]:
df_remove_duplicated_neurotoxic, df_errors_neurotoxic, df_unique_neurotoxic = processing_duplicated(df_neurotoxic, group_seq="sequence", sort_key="label")

In [11]:
df_full_hemotoxic = pd.concat([df_unique_hemotoxic, df_remove_duplicated_hemotoxic])
df_full_cytotoxic = pd.concat([df_remove_duplicated_cytotoxic, df_unique_cytotoxic])
df_full_neurotoxic = pd.concat([df_remove_duplicated_neurotoxic, df_unique_neurotoxic])

df_full = pd.concat([df_full_cytotoxic, df_full_hemotoxic, df_full_neurotoxic])
df_errors = pd.concat([df_errors_cytotoxic, df_errors_hemotoxic, df_errors_neurotoxic])

- Working with metada

In [12]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [13]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2025,
 'last update date': datetime.datetime(2025, 5, 19, 0, 0),
 'download date': Timestamp('2025-10-17 00:00:00'),
 'file format': 'csv',
 'peptide property': 'toxins, neurotoxic, hemolytic, cytotoxic, toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from uniprot',
 'repository or server': 'https://github.com/cosylabiiit/MultiTox/tree/main/Data',
 'publication': 'https://www.sciencedirect.com/science/article/pii/S0141813025079565',
 'number_of_raw_sequences': 3052,
 'number_of_sequences_retained': 1898,
 'number_of_positive_sequences': 1898,
 'number_of_negative_sequences': 0,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [14]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [15]:
df_full_neurotoxic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_neurotoxic_dataset.csv", index=False)
df_full_cytotoxic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_cytotoxic_dataset.csv", index=False)
df_full_hemotoxic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)

df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)